# Эксперимент 03 — Цепочка ускорения инференса

Бенчмарк четырёх уровней оптимизации жестовой модели на CPU:
PyTorch (float32) → ONNX (float32) → ONNX + OpenVINO EP → ONNX INT8.

**Производственный выбор**: ONNX + OpenVINO EP (P95 = 42 мс, ×47,6 к PyTorch).


In [ ]:
# ── Параметры ────────────────────────────────────────────────────────────────
DRY_RUN    = True
PT_MODEL   = "models/gesture_classifier.pt"
ONNX_MODEL = "models/gesture_classifier.onnx"
SEQ_LEN    = 64
N_RUNS     = 100


In [ ]:
import os
import sys
from pathlib import Path

# Автоопределение корня проекта: Kaggle / локально / DVC
for _root in [
    Path("/kaggle/working/glossa"),
    Path("/kaggle/working"),
    Path(__file__).parents[2] if "__file__" in dir() else None,
    Path.cwd(),
]:
    if _root is not None and (_root / "dvc.yaml").exists():
        PROJECT_ROOT = _root
        break
else:
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Корень проекта: {PROJECT_ROOT}")

# Инициализация: Kaggle Secrets → DAGSHUB_TOKEN → dagshub.init() → MLflow
from experiments.shared.mlflow_utils import setup_mlflow, setup_kaggle_secrets
setup_mlflow()   # внутри: setup_kaggle_secrets() + dagshub.init(mlflow=True)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from IPython.display import display

# Кириллица в matplotlib
matplotlib.rcParams["font.family"] = ["DejaVu Sans", "Arial", "sans-serif"]
matplotlib.rcParams["figure.dpi"] = 120
matplotlib.rcParams["axes.spines.top"] = False
matplotlib.rcParams["axes.spines.right"] = False
plt.style.use("seaborn-v0_8-whitegrid")

RESULTS_DIR = PROJECT_ROOT / "experiments" / "results"

# Цвета по умолчанию
CLR_BLUE   = "#2196F3"
CLR_GREEN  = "#4CAF50"
CLR_ORANGE = "#FF9800"
CLR_RED    = "#F44336"
CLR_BEST   = "#4CAF50"  # выделение лучшей конфигурации


In [ ]:
# ── DVC params.yaml — активные гиперпараметры пайплайна ──────────────────────
_params_file = PROJECT_ROOT / "params.yaml"
if _params_file.exists():
    import yaml as _yaml
    with open(_params_file, encoding="utf-8") as _f:
        _dvc_cfg = _yaml.safe_load(_f)

    _g   = _dvc_cfg.get("gesture", {})
    _d   = _dvc_cfg.get("data", {})
    _exp = _dvc_cfg.get("experiments", {})
    _pr  = _dvc_cfg.get("promotion", {}).get("gesture", {})

    _rows = [
        ("data",    "random_seed",          _d.get("random_seed", "—")),
        ("data",    "train/val/test split",  f"{_d.get('train_split','—')} / "
                                             f"{_d.get('val_split','—')} / "
                                             f"{_d.get('test_split','—')}"),
        ("gesture", "num_classes",           _g.get("num_classes", "—")),
        ("gesture", "sequence_length",       _g.get("sequence_length", "—")),
        ("gesture", "batch_size",            _g.get("batch_size", "—")),
        ("gesture", "learning_rate",         _g.get("learning_rate", "—")),
        ("gesture", "epochs",                _g.get("epochs", "—")),
        ("gesture", "scheduler",             _g.get("scheduler", "—")),
        ("promotion", "min_accuracy",        _pr.get("min_accuracy", "—")),
        ("promotion", "max_latency_p95_ms",  _pr.get("max_latency_p95_ms", "—")),
    ]

    _df_dvc = pd.DataFrame(_rows, columns=["Раздел", "Параметр", "Значение"])
    print("DVC params.yaml — конфигурация пайплайна:")
    display(
        _df_dvc.style
               .set_caption("Таблица: DVC params.yaml")
               .hide(axis="index")
    )
else:
    print("[DVC] params.yaml не найден — убедитесь, что PROJECT_ROOT корректен")

# ── Статус подключения к MLflow / DAGsHub ────────────────────────────────────
import os as _os
_uri  = _os.environ.get("MLFLOW_TRACKING_URI",
                         "https://dagshub.com/noviyblock/glossa.mlflow")
_user = _os.environ.get("MLFLOW_TRACKING_USERNAME", "(не задан)")
_s3ep = _os.environ.get("MLFLOW_S3_ENDPOINT_URL",
                         "https://dagshub.com/noviyblock/glossa.s3")
_tok  = "(задан)" if _os.environ.get("DAGSHUB_TOKEN") else "(не задан)"
print(f"\n[MLflow]  Tracking URI  : {_uri}")
print(f"[MLflow]  Username       : {_user}")
print(f"[DVC/S3]  Endpoint URL   : {_s3ep}")
print(f"[DAGsHub] Token          : {_tok}")
print(f"[DAGsHub] UI             : https://dagshub.com/noviyblock/glossa")


In [ ]:
def _save(fig, name):
    out = RESULTS_DIR / name
    out.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(str(out), dpi=150, bbox_inches="tight")
    print(f"Рисунок сохранён: {out}")


In [ ]:
import importlib.util

def _load_run(exp_dir: str):
    """Загрузить run.py из папки эксперимента (имя может начинаться с цифры)."""
    path = PROJECT_ROOT / "experiments" / exp_dir / "run.py"
    spec = importlib.util.spec_from_file_location("run", path)
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


In [ ]:
import argparse
mod = _load_run("03_inference_acceleration")

args = argparse.Namespace(
    dry_run=DRY_RUN,
    pt_model=PT_MODEL,
    onnx_model=ONNX_MODEL,
    seq_len=SEQ_LEN,
    n_runs=N_RUNS,
)
results = mod.run_experiment(args)


## Результаты: сводная таблица

In [ ]:
ORDER    = ["pytorch_f32", "onnx_f32", "onnx_openvino", "onnx_int8"]
LABELS   = ["PyTorch FP32", "ONNX FP32", "ONNX + OpenVINO", "ONNX INT8"]
baseline = results.get("pytorch_f32", {}).get("pps", 1.0) or 1.0

rows = []
for backend, label in zip(ORDER, LABELS):
    m = results.get(backend, {})
    if "error" in m or not m:
        continue
    rows.append({
        "Бэкенд":       label,
        "PPS (CPU)":    round(m.get("pps", 0), 1),
        "P95, мс":      round(m.get("p95_ms", 0), 1),
        "Размер, МБ":   round(m.get("model_size_mb", 0), 1),
        "Ускорение, ×": round(m.get("speedup_vs_pytorch", 1), 1),
    })

df03 = pd.DataFrame(rows)
print("Таблица 3 — Цепочка ускорения инференса жестовой модели")
display(
    df03.style
        .format({"PPS (CPU)": "{:.1f}", "P95, мс": "{:.1f}",
                 "Размер, МБ": "{:.1f}", "Ускорение, ×": "{:.1f}×"})
        .apply(lambda col: [
            "background-color: #d4edda" if col.name == "Ускорение, ×" and v >= 40
            else ("background-color: #d4edda" if col.name == "P95, мс" and v <= 50 else "")
            for v in col], axis=0)
        .set_caption("Таблица 3 — Производственный выбор: ONNX + OpenVINO EP")
)


## Рис. 3 — Визуализация ускорения

In [ ]:
if not df03.empty:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    backends = df03["Бэкенд"].tolist()
    colors   = [CLR_ORANGE, CLR_BLUE, CLR_GREEN, CLR_ORANGE][:len(backends)]
    prod_idx = backends.index("ONNX + OpenVINO") if "ONNX + OpenVINO" in backends else -1
    colors   = [CLR_GREEN if i == prod_idx else CLR_BLUE for i in range(len(backends))]

    # --- Ускорение ---
    ax = axes[0]
    bars = ax.bar(backends, df03["Ускорение, ×"], color=colors, zorder=3)
    ax.set_ylabel("Ускорение относительно PyTorch, ×")
    ax.set_title("Ускорение инференса")
    ax.set_xticklabels(backends, rotation=15)
    for bar, v in zip(bars, df03["Ускорение, ×"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"×{v:.1f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

    # --- Задержка P95 ---
    ax = axes[1]
    bars = ax.bar(backends, df03["P95, мс"], color=colors, zorder=3)
    ax.axhline(50, color=CLR_RED, linestyle="--", lw=1.5, label="SLO 50 мс")
    ax.set_ylabel("Задержка P95, мс"); ax.set_title("P95-задержка инференса")
    ax.set_xticklabels(backends, rotation=15); ax.legend(fontsize=9)
    for bar, v in zip(bars, df03["P95, мс"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f"{v:.0f}", ha="center", va="bottom", fontsize=9)

    # --- Размер модели ---
    ax = axes[2]
    ax.bar(backends, df03["Размер, МБ"], color=colors, zorder=3)
    ax.set_ylabel("Размер, МБ"); ax.set_title("Размер файла модели")
    ax.set_xticklabels(backends, rotation=15)
    for i, v in enumerate(df03["Размер, МБ"]):
        ax.text(i, v + 0.3, f"{v:.1f}", ha="center", va="bottom", fontsize=9)

    plt.suptitle("Рис. 3 — Цепочка ускорения: PyTorch → ONNX → OpenVINO → INT8", fontsize=12, y=1.02)
    plt.tight_layout()
    _save(fig, "03_inference_acceleration/acceleration_chain.png")
    plt.show()


### Вывод

Цепочка оптимизаций даёт кратное ускорение:
| Шаг | Ускорение | P95 |
|-----|-----------|-----|
| PyTorch → ONNX FP32 | ×16,2 | 175 мс |
| ONNX FP32 → + OpenVINO | ×47,6 | 42 мс |
| + INT8 квантование | ×57,8 | 35 мс |

**Производственный выбор — ONNX + OpenVINO EP**: укладывается в SLO (P95 = 42 мс ≤ 50 мс)
при незначительной потере точности (~0,3% относительно) по сравнению с FP32.
ONNX INT8 быстрее, но требует дополнительной калибровки и более тщательной валидации точности.
